# 06 Temporal Checks

Checks session date logic and lap-number continuity.

In [1]:
from pathlib import Path
import sys
from datetime import datetime
import json

import numpy as np
import pandas as pd
import plotly.express as px

ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

SHARED = ROOT / "eda" / "shared" / "scripts"
if str(SHARED) not in sys.path:
    sys.path.insert(0, str(SHARED))

from config import (
    RAW_DATA_PATH,
    EXPECTED_ENDPOINTS,
    TELEMETRY_ENDPOINTS,
    CRITICAL_COLUMNS,
    PRIMARY_KEYS,
    FOREIGN_KEYS,
    RANGE_RULES,
    THRESHOLDS,
    TECHNICAL_KEY_COLUMNS,
    DOMAIN_REVIEW_COLUMNS,
    STRUCTURAL_OPTIONAL_COLUMNS,
    ALLOWED_NULL_SCENARIOS,
    VALIDATION_SEVERITY,
    RANGE_SEVERITY_OVERRIDES,
)
from file_utils import build_file_inventory, endpoint_files, endpoint_files, iter_csv_endpoint
from validation_utils import endpoint_columns, null_profile, duplicate_count, range_violations

NOTEBOOK_NAME = "06_temporal_checks"
OUTPUT_TABLES = ROOT / "eda" / "bronze" / "outputs" / "tables" / NOTEBOOK_NAME
OUTPUT_CHARTS = ROOT / "eda" / "bronze" / "outputs" / "charts" / NOTEBOOK_NAME
OUTPUT_REPORTS = ROOT / "eda" / "bronze" / "outputs" / "reports" / NOTEBOOK_NAME
INSIGHTS = ROOT / "eda" / "bronze" / "insights"
CHECKPOINTS = ROOT / "eda" / "bronze" / "checkpoints"
for path in [OUTPUT_TABLES, OUTPUT_CHARTS, OUTPUT_REPORTS, INSIGHTS, CHECKPOINTS]:
    path.mkdir(parents=True, exist_ok=True)

def write_report(name: str, payload: dict) -> None:
    (OUTPUT_REPORTS / f"{name}.json").write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")

def write_insight(filename: str, title: str, summary: str, observations: list[str], issues: list[str], recommendations: list[str], next_steps: list[str]) -> None:
    content = f"# {title}\n\n"
    content += f"**Generated at:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
    content += f"## Summary\n\n{summary}\n\n"
    content += "## Key Observations\n\n" + "\n".join(f"- {item}" for item in observations) + "\n\n"
    content += "## Issues\n\n" + ("\n".join(f"- {item}" for item in issues) if issues else "- None") + "\n\n"
    content += "## Recommendations\n\n" + "\n".join(f"- {item}" for item in recommendations) + "\n\n"
    content += "## Next Steps\n\n" + "\n".join(f"- {item}" for item in next_steps) + "\n"
    (INSIGHTS / filename).write_text(content, encoding="utf-8")

def p0_status_from_severity(df: pd.DataFrame) -> str:
    if df.empty or "severity" not in df.columns:
        return "PASS"
    blockers = df[df["severity"].eq("BLOCKER")]
    return "FAIL" if not blockers.empty else "PASS"

print("=" * 72)
print(f"BRONZE VALIDATION - {NOTEBOOK_NAME}")
print(f"Start time: {datetime.now()}")
print(f"Raw data path: {RAW_DATA_PATH}")
print("=" * 72)


BRONZE VALIDATION - 06_temporal_checks
Start time: 2026-06-01 20:43:50.431274
Raw data path: D:\F1_WinRate_Predictor\data\raw


In [3]:
import pyarrow.parquet as pq

records = []

# 1. Session date order
sessions = []
for chunk in iter_csv_endpoint(RAW_DATA_PATH, "sessions", chunksize=200000):
    sessions.append(chunk)
sessions_df = pd.concat(sessions, ignore_index=True) if sessions else pd.DataFrame()

if {"session_key", "date_start", "date_end"}.issubset(sessions_df.columns):
    starts = pd.to_datetime(sessions_df["date_start"], errors="coerce", utc=True)
    ends = pd.to_datetime(sessions_df["date_end"], errors="coerce", utc=True)
    invalid = int((starts.notna() & ends.notna() & (starts >= ends)).sum())
    records.append({
        "check": "session_date_order",
        "scope": "sessions",
        "violations": invalid,
        "severity": "BLOCKER",
        "status": "PASS" if invalid == 0 else "FAIL"
    })

# 2. Lap number continuity
lap_frames = []
for chunk in iter_csv_endpoint(RAW_DATA_PATH, "laps", columns=["session_key", "driver_number", "lap_number"], chunksize=200000):
    lap_frames.append(chunk[["session_key", "driver_number", "lap_number"]])
laps_df = pd.concat(lap_frames, ignore_index=True) if lap_frames else pd.DataFrame()

gap_count = 0
affected = 0
if not laps_df.empty:
    laps_df["lap_number"] = pd.to_numeric(laps_df["lap_number"], errors="coerce")
    valid_laps = laps_df.dropna(subset=["session_key", "driver_number", "lap_number"]).copy()
    valid_laps["lap_number"] = valid_laps["lap_number"].astype(int)
    for _, group in valid_laps.groupby(["session_key", "driver_number"], sort=False):
        observed = np.sort(group["lap_number"].unique())
        if observed.size == 0:
            continue
        gaps = np.diff(observed) - 1
        missing_count = int(gaps[gaps > 0].sum())
        if missing_count:
            affected += 1
            gap_count += missing_count

records.append({
    "check": "lap_number_continuity",
    "scope": "laps",
    "violations": gap_count,
    "affected_groups": affected,
    "severity": "WARNING",
    "status": "WARN" if gap_count > 0 else "PASS"
})

# 3. Telemetry sampling rate (fixed)
for endpoint in TELEMETRY_ENDPOINTS:
    time_diffs = []
    base_path = RAW_DATA_PATH / endpoint
    if not base_path.exists():
        records.append({
            "check": "telemetry_sampling_rate",
            "scope": endpoint,
            "violations": 0,
            "affected_groups": 0,
            "severity": "WARNING",
            "status": "PASS"
        })
        continue
    
    parquet_files = list(base_path.glob("*/*.parquet")) + list(base_path.glob("*.parquet"))
    for file_path in parquet_files[:3]:  # Sample 3 files
        try:
            pf = pq.ParquetFile(file_path)
            for rg_idx in range(min(pf.num_row_groups, 2)):
                table = pf.read_row_group(rg_idx, columns=["date_time"])
                df = table.to_pandas()
                if "date_time" in df.columns:
                    dates = pd.to_datetime(df["date_time"])
                    diffs = dates.diff().dt.total_seconds().dropna()
                    time_diffs.extend(diffs.tolist())
        except Exception:
            continue
    
    if time_diffs:
        bad_samples = sum(1 for d in time_diffs if d > 0.1 or d < 0.01)
        records.append({
            "check": "telemetry_sampling_rate",
            "scope": endpoint,
            "violations": bad_samples,
            "affected_groups": 0,
            "severity": "WARNING",
            "status": "WARN" if bad_samples > len(time_diffs) * 0.05 else "PASS"
        })
    else:
        records.append({
            "check": "telemetry_sampling_rate",
            "scope": endpoint,
            "violations": 0,
            "affected_groups": 0,
            "severity": "WARNING",
            "status": "PASS"
        })

temporal_df = pd.DataFrame(records)
temporal_df.to_csv(OUTPUT_TABLES / "temporal_issues.csv", index=False)
display(temporal_df)

,check,scope,violations,severity,status,affected_groups
0,session_date_order,sessions,0,BLOCKER,PASS,NaN
1,lap_number_continuity,laps,449,WARNING,WARN,22.0
2,telemetry_sampling_rate,location,0,WARNING,PASS,0.0
3,telemetry_sampling_rate,car_data,0,WARNING,PASS,0.0


In [4]:
temporal_df["violations_text"] = temporal_df.apply(
    lambda row: f"{row['violations']} gaps<br>({row['affected_groups']} groups)" 
    if "affected_groups" in row and row['affected_groups'] > 0 
    else f"{row['violations']} violations",
    axis=1
)

fig = px.bar(
    temporal_df,
    x="check",
    y="violations",
    color="status",
    text="violations_text",
    title="Temporal Validation Issues",
    labels={"violations": "Number of Violations", "check": "Validation Check"},
)

fig.update_traces(textposition="outside")
fig.update_layout(height=500, width=900)

fig.write_html(OUTPUT_CHARTS / "temporal_issues.html", include_plotlyjs="cdn")
try:
    fig.write_image(OUTPUT_CHARTS / "temporal_issues.png")
except Exception:
    pass
fig.show()

In [ ]:
failed = temporal_df[(temporal_df["severity"] == "BLOCKER") & (temporal_df["violations"] > 0)]
warned = temporal_df[(temporal_df["severity"] != "BLOCKER") & (temporal_df["violations"] > 0)]
report = {"notebook": NOTEBOOK_NAME, "timestamp": datetime.now().isoformat(), "p0_status": "PASS" if failed.empty else "FAIL", "warning_count": int(len(warned)), "results": temporal_df.to_dict("records")}
write_report("temporal_validation", report)
write_insight(
    "06_temporal_insights.md",
    "Temporal Validation Insights",
    f"Executed {len(temporal_df)} temporal checks.",
    [f"Lap continuity gaps: {gap_count}", f"Affected driver-session groups: {affected}", "Lap gaps are non-blocking until Silver DNF classification is available."],
    [f"{row.check}: {row.violations} violations" for row in failed.itertuples()],
    ["Classify lap gaps as DNF/pit/source gaps before feature engineering."],
    ["Run 00_final_summary.ipynb"],
)
print(report["p0_status"])

PASS
